# ToTTI T4 — fine-tune Opus-MT en-it on the distilled pairs

Decision 0038, plan section T4. Runtime: **T4 GPU** (Runtime → Change runtime type).
Expected cost: **~1–2 compute units per run** (the model is 78M parameters, ~4M tokens
per epoch, a few minutes per epoch; early stopping usually ends the run well before the
10-epoch ceiling).

**Data policy.** The only data uploaded here is the upload bundle built at home: the
AMI/ICSI-derived distilled pairs (CC BY 4.0 public-corpus derivatives — fine to upload to
your own Colab session) and the FLORES-200 devtest files (CC BY-SA 4.0). The private test
set **never** goes to Colab: the accepting scores run at home with `score_at_home.py`.

Steps: run all cells top to bottom, upload `totti-t4-bundle.zip` when asked, download
`opus-mt-en-it-totti-ct2.zip` at the end.

In [ ]:
# A T4 (or better) must show up here; otherwise fix the runtime type before going on.
!nvidia-smi

In [ ]:
# torch ships with Colab; the rest is small. ctranslate2 is only for the final conversion.
%pip install -q "transformers>=4.49,<5" "sacrebleu>=2.4" pyyaml sentencepiece \
    "ctranslate2>=4.5" accelerate

In [ ]:
# Upload the bundle built at home (see training/totti/README.md):
#   totti-t4-bundle.zip — config.yaml, train.py, convert.sh,
#   data/{clean-train.jsonl, clean-dev.jsonl, eng_Latn.devtest, ita_Latn.devtest}
from google.colab import files

uploaded = files.upload()
!unzip -o -q totti-t4-bundle.zip -d /content/totti
!ls -R /content/totti

In [ ]:
# Only if the FLORES files were left out of the bundle: fetch the official Meta archive.
import os

if not os.path.exists("/content/totti/data/eng_Latn.devtest"):
    !wget -q https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz
    !tar xzf flores200_dataset.tar.gz
    !cp flores200_dataset/devtest/eng_Latn.devtest /content/totti/data/
    !cp flores200_dataset/devtest/ita_Latn.devtest /content/totti/data/
!wc -l /content/totti/data/*

In [ ]:
# Training: early stopping on dev chrF, then the FLORES guardrail verdict at the end.
# Exit code 1 means the FLORES check FAILED — do not ship that checkpoint.
!cd /content/totti && python train.py --config config.yaml --out out

In [ ]:
# CTranslate2 int8 conversion, ready to drop into data/models/ at home.
!cd /content/totti && bash convert.sh out/best out/opus-mt-en-it-totti-ct2
!ls /content/totti/out/opus-mt-en-it-totti-ct2

In [ ]:
# Download the converted model (and the fp16 checkpoint too, in case of re-conversion).
!cd /content/totti/out && zip -r -q /content/opus-mt-en-it-totti-ct2.zip opus-mt-en-it-totti-ct2
from google.colab import files

files.download("/content/opus-mt-en-it-totti-ct2.zip")

## At home

Unzip into `data/models/opus-mt-en-it-totti-ct2/` and run the accepting scores (the
private test set stays on the Mac):

```bash
uv run python training/totti/score_at_home.py data/models/opus-mt-en-it-totti-ct2
```

Accepted only if domain chrF beats **60.3** and FLORES stays within **1.0** of **57.6**.